# Q4: Feature Engineering

**Phase 5:** Feature Engineering & Aggregation  
**Points: 9 points**

**Focus:** Create derived features, perform time-based aggregations,
calculate rolling windows.

**Lecture Reference:** See **Lecture 11, Notebook 2**
(`11/demo/02_wrangling_feature_engineering.ipynb`), Phase 5 for examples
of feature engineering, time-based aggregations, and rolling window
calculations. Also see **Lecture 09** for time series rolling window
operations.

------------------------------------------------------------------------

## Objective

Create derived features, perform time-based aggregations, and calculate
rolling windows for time series analysis.

**Time Series Note:** Rolling windows are essential for time series
data. They capture temporal dependencies (e.g., 7-hour rolling mean
captures short-term patterns). See **Lecture 09** for time series
rolling window operations. For hourly data, common window sizes are 7-24
hours (capturing daily patterns). Use pandas `rolling()` method with
`window` parameter to specify the number of periods.

------------------------------------------------------------------------

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q4_features.csv`

**Format:** CSV file **Content:** Dataset with all derived features
added **Requirements:** - All original columns from Q3 - All new derived
features added as columns - **No index column** (save with
`index=False`)

### 2. `output/q4_rolling_features.csv`

**Format:** CSV file **Content:** Dataset with rolling window features
**Required Columns:** - Original datetime column - At least one rolling
window calculation column (e.g., `water_temp_rolling_7h`,
`air_temp_rolling_24h`)

**Requirements:** - Must include at least one rolling window
calculation - Rolling window names should be descriptive (e.g.,
`temp_rolling_7h` for 7-hour rolling mean) - **No index column** (save
with `index=False`)

**Example columns:**

``` csv
Measurement Timestamp,wind_speed_rolling_7h,humidity_rolling_24h,pressure_rolling_7h
2022-01-01 00:00:00,6.8,65.2,1013.5
2022-01-01 01:00:00,6.9,65.3,1013.6
...
```

**Note:** The example shows rolling windows of predictor variables (wind
speed, humidity, pressure), not the target variable. If you’re
predicting Air Temperature, do NOT create rolling windows of Air
Temperature - this causes data leakage.

### 3. `output/q4_feature_list.txt`

**Format:** Plain text file **Content:** List of new features created
(one per line) **Requirements:** - One feature name per line - No extra
text, just feature names - Include all derived features, rolling
features, and categorical features created

**Example format:**

    temp_difference
    temp_ratio
    wind_speed_squared
    comfort_index
    water_temp_rolling_7h
    air_temp_rolling_24h
    wind_speed_rolling_7h
    temp_category
    wind_category

------------------------------------------------------------------------

## Requirements Checklist

- [ ] Derived features created (differences, ratios, interactions, etc.)
- [ ] Time-based aggregations performed (by hour, day, month, etc.) -
  optional but recommended
- [ ] At least one rolling window calculation (rolling mean, rolling
  median, etc.)
- [ ] Categorical features created (if applicable)
- [ ] Feature list documented
- [ ] All 3 required artifacts saved with exact filenames

------------------------------------------------------------------------

## Your Approach

1.  **Create derived features:**

    ``` python
    # Examples:
    df['temp_difference'] = df['Air Temperature'] - df['Water Temperature']
    df['temp_ratio'] = df['Air Temperature'] / (df['Water Temperature'] + 0.1)  # Avoid division by zero
    df['wind_speed_squared'] = df['Wind Speed'] ** 2
    df['comfort_index'] = (df['Air Temperature'] * 0.4 + 
                          (100 - df['Humidity']) * 0.3 + 
                          (20 - df['Wind Speed']) * 0.3)
    ```

2.  **Calculate rolling windows:**

    ``` python
    # Ensure datetime index is set and sorted
    df = df.sort_index()

    # Rolling windows (examples - use predictor variables only, not your target variable)
    df['wind_speed_rolling_7h'] = df['Wind Speed'].rolling(window=7, min_periods=1).mean()
    df['wind_speed_rolling_24h'] = df['Wind Speed'].rolling(window=24, min_periods=1).mean()
    df['humidity_rolling_7h'] = df['Humidity'].rolling(window=7, min_periods=1).mean()
    df['pressure_rolling_7h'] = df['Barometric Pressure'].rolling(window=7, min_periods=1).mean()
    ```

    ⚠️ **Important:** Only create rolling windows of **predictor
    variables**, not your target variable. Creating rolling windows of
    the target variable causes data leakage (you’d be predicting the
    target from a smoothed version of itself). See Q6 and Q7 for more
    details on avoiding data leakage.

3.  **Create categorical features (optional):**

    ``` python
    df['temp_category'] = pd.cut(df['Air Temperature'], 
                                 bins=[-np.inf, 10, 20, 30, np.inf],
                                 labels=['Cold', 'Cool', 'Warm', 'Hot'])
    ```

4.  **Check for infinity values:**

    - After creating derived features (especially ratios), check for
      infinity values:

    ``` python
    # Replace infinity with NaN, then handle appropriately
    df = df.replace([np.inf, -np.inf], np.nan)
    # Fill or drop as needed based on your analysis
    ```

5.  **Document and save:**

    - Save all features:
      `df.reset_index().to_csv('output/q4_features.csv', index=False)`
      - **Important:** When saving CSVs with datetime index, use
        `reset_index()` to convert index to column, otherwise it will
        not be included in the output.
    - Save rolling features:
      `df[['rolling_col1', 'rolling_col2', ...]].reset_index().to_csv('output/q4_rolling_features.csv', index=False)`
      - **Important:** Remember to use `reset_index()` before saving to
        include the datetime as a column.
    - Write feature list:
      `with open('output/q4_feature_list.txt', 'w') as f: f.write('\n'.join(feature_names))`

------------------------------------------------------------------------

## Decision Points

- **Derived features:** What relationships might be useful? Temperature
  differences? Ratios? Interactions between variables?
- **Rolling windows:** What window size makes sense? 7 hours? 24 hours?
  Consider the temporal scale of your data. For hourly data, 7-24 hours
  captures daily patterns.
- **Time-based aggregations:** Aggregate by hour? Day? Week? What
  temporal granularity is useful for your analysis?

------------------------------------------------------------------------

## Checkpoint

After Q4, you should have: - \[ \] Derived features created - \[ \] At
least one rolling window calculation - \[ \] Feature list documented -
\[ \] All 3 artifacts saved: `q4_features.csv`,
`q4_rolling_features.csv`, `q4_feature_list.txt`

------------------------------------------------------------------------

**Next:** Continue to `q5_pattern_analysis.md` for Pattern Analysis.

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

# Load wrangled data from Q3
df = pd.read_csv('output/q3_wrangled_data.csv', parse_dates=['Measurement Timestamp'], index_col='Measurement Timestamp')
# Or if you saved without index:
# df = pd.read_csv('output/q3_wrangled_data.csv')
# df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
# df = df.set_index('Measurement Timestamp')
print(f"Loaded {len(df):,} records with datetime index")

Loaded 195,892 records with datetime index


In [ ]:
# Load wrangled data from Q3 with datetime index
# Adjust the datetime column name based on your data
datetime_col = 'Measurement Timestamp'  # Update if different

df = pd.read_csv('output/q3_wrangled_data.csv', 
                 parse_dates=[datetime_col], 
                 index_col=datetime_col)

print(f"\nLoaded {len(df):,} records with datetime index")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

# Display first few rows
print("\nFirst few rows:")
print(df.head())


Loaded 195,892 records with datetime index
Columns: 25
Date range: 2015-04-25 09:00:00 to 2025-11-24 12:00:00

First few rows:
                                      Station Name  Air Temperature  \
Measurement Timestamp                                                 
2015-04-25 09:00:00    63rd Street Weather Station             7.00   
2015-04-30 05:00:00    63rd Street Weather Station             6.10   
2015-05-22 15:00:00     Oak Street Weather Station              NaN   
2015-05-22 16:00:00         Foster Weather Station             9.17   
2015-05-22 17:00:00         Foster Weather Station             9.28   

                       Wet Bulb Temperature  Humidity  Rain Intensity  \
Measurement Timestamp                                                   
2015-04-25 09:00:00                     5.9        86             7.2   
2015-04-30 05:00:00                     4.3        76             0.0   
2015-05-22 15:00:00                     7.0        55             0.0   
2015-05-2

In [6]:
# Identify numeric columns (excluding temporal features we created in Q3)
temporal_cols = ['hour', 'day_of_week', 'month', 'year', 'day_name', 'is_weekend', 
                'day_of_month', 'quarter']
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove temporal features from numeric columns
sensor_cols = [col for col in numeric_cols if col not in temporal_cols]

print(f"\nSensor columns (for feature engineering): {sensor_cols}")



Sensor columns (for feature engineering): ['Air Temperature', 'Wet Bulb Temperature', 'Humidity', 'Rain Intensity', 'Interval Rain', 'Total Rain', 'Precipitation Type', 'Wind Direction', 'Wind Speed', 'Maximum Wind Speed', 'Barometric Pressure', 'Solar Radiation', 'Heading', 'Battery Life']


In [7]:

# ========================================
# STEP 2: CREATE DERIVED FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 1: CREATING DERIVED FEATURES")
print("="*60)

# Track all new features created
new_features = []

print("\nCreating derived features...")

# Example 1: Temperature-based features (adjust column names to your data)
# These are examples - adjust based on your actual column names
if 'Air Temperature' in df.columns and 'Water Temperature' in df.columns:
    print("\nTemperature-based features:")
    
    # Temperature difference
    df['temp_difference'] = df['Air Temperature'] - df['Water Temperature']
    new_features.append('temp_difference')
    print("  ✓ temp_difference: Air Temperature - Water Temperature")
    
    # Temperature ratio (avoid division by zero)
    df['temp_ratio'] = df['Air Temperature'] / (df['Water Temperature'] + 0.1)
    new_features.append('temp_ratio')
    print("  ✓ temp_ratio: Air Temperature / Water Temperature")
    
    # Average temperature
    df['temp_average'] = (df['Air Temperature'] + df['Water Temperature']) / 2
    new_features.append('temp_average')
    print("  ✓ temp_average: Average of air and water temperature")

# Example 2: Wind-based features
if 'Wind Speed' in df.columns:
    print("\nWind-based features:")
    
    # Wind speed squared (kinetic energy proxy)
    df['wind_speed_squared'] = df['Wind Speed'] ** 2
    new_features.append('wind_speed_squared')
    print("  ✓ wind_speed_squared: Wind Speed squared")
    
    # Wind speed categories
    df['wind_category'] = pd.cut(df['Wind Speed'], 
                                  bins=[-np.inf, 5, 10, 15, np.inf],
                                  labels=['Calm', 'Light', 'Moderate', 'Strong'])
    new_features.append('wind_category')
    print("  ✓ wind_category: Categorical wind speed")

# Example 3: Comfort/Feel index (if you have temp, humidity, wind)
if all(col in df.columns for col in ['Air Temperature', 'Humidity', 'Wind Speed']):
    print("\nComfort index:")
    
    # Simple comfort index (weighted combination)
    df['comfort_index'] = (df['Air Temperature'] * 0.4 + 
                          (100 - df['Humidity']) * 0.3 + 
                          (20 - df['Wind Speed']) * 0.3)
    new_features.append('comfort_index')
    print("  ✓ comfort_index: Weighted comfort metric")

# Example 4: Humidity-based features
if 'Humidity' in df.columns:
    print("\nHumidity-based features:")
    
    # Humidity squared
    df['humidity_squared'] = df['Humidity'] ** 2
    new_features.append('humidity_squared')
    print("  ✓ humidity_squared: Humidity squared")

# Example 5: Interaction features
if 'Air Temperature' in df.columns and 'Wind Speed' in df.columns:
    print("\nInteraction features:")
    
    # Wind chill approximation (temp * wind interaction)
    df['temp_wind_interaction'] = df['Air Temperature'] * df['Wind Speed']
    new_features.append('temp_wind_interaction')
    print("  ✓ temp_wind_interaction: Temperature × Wind Speed")

# Example 6: Pressure-based features (if available)
if 'Barometric Pressure' in df.columns:
    print("\nPressure-based features:")
    
    # Pressure deviation from mean
    pressure_mean = df['Barometric Pressure'].mean()
    df['pressure_deviation'] = df['Barometric Pressure'] - pressure_mean
    new_features.append('pressure_deviation')
    print(f"  ✓ pressure_deviation: Deviation from mean ({pressure_mean:.2f})")

# Check for infinity values after creating derived features
print("\nChecking for infinity values...")
inf_mask = np.isinf(df.select_dtypes(include=[np.number])).any(axis=1)
inf_count = inf_mask.sum()
if inf_count > 0:
    print(f"  Warning: Found {inf_count} rows with infinity values")
    print("  Replacing infinity with NaN...")
    df = df.replace([np.inf, -np.inf], np.nan)
    print("  ✓ Infinity values replaced with NaN")
else:
    print("  ✓ No infinity values found")

print(f"\nTotal derived features created: {len(new_features)}")


STEP 1: CREATING DERIVED FEATURES

Creating derived features...

Wind-based features:
  ✓ wind_speed_squared: Wind Speed squared
  ✓ wind_category: Categorical wind speed

Comfort index:
  ✓ comfort_index: Weighted comfort metric

Humidity-based features:
  ✓ humidity_squared: Humidity squared

Interaction features:
  ✓ temp_wind_interaction: Temperature × Wind Speed

Pressure-based features:
  ✓ pressure_deviation: Deviation from mean (994.31)

Checking for infinity values...
  ✓ No infinity values found

Total derived features created: 6


In [8]:
# ========================================
# STEP 3: CALCULATE ROLLING WINDOWS
# ========================================
print("\n" + "="*60)
print("STEP 2: CALCULATING ROLLING WINDOW FEATURES")
print("="*60)

# Ensure data is sorted by datetime index (critical for rolling windows!)
print("\nSorting data by datetime index...")
df = df.sort_index()
print("✓ Data sorted chronologically")

print("\nCalculating rolling window features...")
print("Note: Using predictor variables only (NOT the target variable)")

rolling_features = []

# Identify which columns to create rolling features for
# DO NOT include your target variable here!
# For example, if predicting Air Temperature, don't include it in rolling features

# Rolling windows for Wind Speed
if 'Wind Speed' in df.columns:
    print("\nWind Speed rolling windows:")
    
    # 7-hour rolling mean
    df['wind_speed_rolling_7h'] = df['Wind Speed'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('wind_speed_rolling_7h')
    print("  ✓ wind_speed_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['wind_speed_rolling_24h'] = df['Wind Speed'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('wind_speed_rolling_24h')
    print("  ✓ wind_speed_rolling_24h: 24-hour rolling mean")
    
    # 7-hour rolling std (volatility)
    df['wind_speed_rolling_std_7h'] = df['Wind Speed'].rolling(window=7, min_periods=1).std()
    rolling_features.append('wind_speed_rolling_std_7h')
    print("  ✓ wind_speed_rolling_std_7h: 7-hour rolling std")

# Rolling windows for Humidity
if 'Humidity' in df.columns:
    print("\nHumidity rolling windows:")
    
    # 7-hour rolling mean
    df['humidity_rolling_7h'] = df['Humidity'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('humidity_rolling_7h')
    print("  ✓ humidity_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['humidity_rolling_24h'] = df['Humidity'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('humidity_rolling_24h')
    print("  ✓ humidity_rolling_24h: 24-hour rolling mean")

# Rolling windows for Water Temperature (if it's a predictor, not target)
if 'Water Temperature' in df.columns:
    print("\nWater Temperature rolling windows:")
    
    # 7-hour rolling mean
    df['water_temp_rolling_7h'] = df['Water Temperature'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('water_temp_rolling_7h')
    print("  ✓ water_temp_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['water_temp_rolling_24h'] = df['Water Temperature'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('water_temp_rolling_24h')
    print("  ✓ water_temp_rolling_24h: 24-hour rolling mean")

# Rolling windows for Barometric Pressure
if 'Barometric Pressure' in df.columns:
    print("\nBarometric Pressure rolling windows:")
    
    # 7-hour rolling mean
    df['pressure_rolling_7h'] = df['Barometric Pressure'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('pressure_rolling_7h')
    print("  ✓ pressure_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['pressure_rolling_24h'] = df['Barometric Pressure'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('pressure_rolling_24h')
    print("  ✓ pressure_rolling_24h: 24-hour rolling mean")

print(f"\nTotal rolling features created: {len(rolling_features)}")

# Add rolling features to new features list
new_features.extend(rolling_features)



STEP 2: CALCULATING ROLLING WINDOW FEATURES

Sorting data by datetime index...
✓ Data sorted chronologically

Calculating rolling window features...
Note: Using predictor variables only (NOT the target variable)

Wind Speed rolling windows:
  ✓ wind_speed_rolling_7h: 7-hour rolling mean
  ✓ wind_speed_rolling_24h: 24-hour rolling mean
  ✓ wind_speed_rolling_std_7h: 7-hour rolling std

Humidity rolling windows:
  ✓ humidity_rolling_7h: 7-hour rolling mean
  ✓ humidity_rolling_24h: 24-hour rolling mean

Barometric Pressure rolling windows:
  ✓ pressure_rolling_7h: 7-hour rolling mean
  ✓ pressure_rolling_24h: 24-hour rolling mean

Total rolling features created: 7


In [9]:

# ========================================
# STEP 4: CREATE CATEGORICAL FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 3: CREATING CATEGORICAL FEATURES")
print("="*60)

categorical_features = []

# Temperature categories (if Air Temperature exists)
if 'Air Temperature' in df.columns:
    print("\nTemperature categories:")
    
    df['temp_category'] = pd.cut(df['Air Temperature'], 
                                bins=[-np.inf, 10, 20, 30, np.inf],
                                labels=['Cold', 'Cool', 'Warm', 'Hot'])
    categorical_features.append('temp_category')
    print("  ✓ temp_category: Cold/Cool/Warm/Hot")
    print(f"    Distribution:\n{df['temp_category'].value_counts()}")

# Time of day categories
if 'hour' in df.columns:
    print("\nTime of day categories:")
    
    df['time_of_day'] = pd.cut(df['hour'], 
                                bins=[-1, 6, 12, 18, 24],
                                labels=['Night', 'Morning', 'Afternoon', 'Evening'])
    categorical_features.append('time_of_day')
    print("  ✓ time_of_day: Night/Morning/Afternoon/Evening")
    print(f"    Distribution:\n{df['time_of_day'].value_counts()}")

# Season categories (if month exists)
if 'month' in df.columns:
    print("\nSeason categories:")
    
    df['season'] = pd.cut(df['month'], 
                        bins=[0, 3, 6, 9, 12],
                        labels=['Winter', 'Spring', 'Summer', 'Fall'])
    categorical_features.append('season')
    print("  ✓ season: Winter/Spring/Summer/Fall")
    print(f"    Distribution:\n{df['season'].value_counts()}")

print(f"\nTotal categorical features created: {len(categorical_features)}")

# Add categorical features to new features list
new_features.extend(categorical_features)



STEP 3: CREATING CATEGORICAL FEATURES

Temperature categories:
  ✓ temp_category: Cold/Cool/Warm/Hot
    Distribution:
temp_category
Cold    80347
Warm    59261
Cool    53161
Hot      3048
Name: count, dtype: int64

Time of day categories:
  ✓ time_of_day: Night/Morning/Afternoon/Evening
    Distribution:
time_of_day
Night        56701
Afternoon    49185
Morning      49162
Evening      40844
Name: count, dtype: int64

Season categories:
  ✓ season: Winter/Spring/Summer/Fall
    Distribution:
season
Summer    56730
Spring    49725
Fall      49331
Winter    40106
Name: count, dtype: int64

Total categorical features created: 3


In [10]:

# ========================================
# STEP 5: HANDLE MISSING VALUES IN NEW FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 4: HANDLING MISSING VALUES IN NEW FEATURES")
print("="*60)

# Check for missing values in new features
print("\nChecking for missing values in new features...")
new_features_df = df[new_features]
missing_in_new = new_features_df.isnull().sum()
missing_any = missing_in_new[missing_in_new > 0]

if len(missing_any) > 0:
    print(f"\nFound missing values in {len(missing_any)} features:")
    for col, count in missing_any.items():
        pct = (count / len(df)) * 100
        print(f"  - {col}: {count} ({pct:.2f}%)")
    
    # Handle missing values (rolling windows often have NaN at the start)
    print("\nHandling missing values...")
    # For rolling features, forward-fill is appropriate
    for col in rolling_features:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(method='bfill')  # Backward fill for initial values
    print("  ✓ Missing values in rolling features handled")
else:
    print("  ✓ No missing values in new features")


STEP 4: HANDLING MISSING VALUES IN NEW FEATURES

Checking for missing values in new features...

Found missing values in 7 features:
  - comfort_index: 75 (0.04%)
  - temp_wind_interaction: 75 (0.04%)
  - pressure_deviation: 146 (0.07%)
  - wind_speed_rolling_std_7h: 1 (0.00%)
  - pressure_rolling_7h: 140 (0.07%)
  - pressure_rolling_24h: 123 (0.06%)
  - temp_category: 75 (0.04%)

Handling missing values...
  ✓ Missing values in rolling features handled


/var/folders/6w/gv8g6zzd417_9g9fl64d_nhc0000gn/T/ipykernel_39222/3971671773.py:25: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='bfill')  # Backward fill for initial values


In [11]:

# ========================================
# SAVE ARTIFACT 1: q4_features.csv
# ========================================
print("\n" + "="*60)
print("SAVING ARTIFACTS")
print("="*60)

print("\nSaving all features...")
# Reset index to save datetime as a column
df_to_save = df.reset_index()
df_to_save.to_csv('output/q4_features.csv', index=False)
print("✓ Saved: output/q4_features.csv")
print(f"  Columns saved: {df_to_save.shape[1]}")
print(f"  Rows saved: {df_to_save.shape[0]}")

# ========================================
# SAVE ARTIFACT 2: q4_rolling_features.csv
# ========================================
print("\nSaving rolling features...")
# Select only rolling features and reset index
rolling_df = df[rolling_features].reset_index()
rolling_df.to_csv('output/q4_rolling_features.csv', index=False)
print("✓ Saved: output/q4_rolling_features.csv")
print(f"  Columns saved: {rolling_df.shape[1]}")
print(f"  Rolling features: {len(rolling_features)}")

# Display sample
print("\nSample of rolling features:")
print(rolling_df.head())

# ========================================
# SAVE ARTIFACT 3: q4_feature_list.txt
# ========================================
print("\nSaving feature list...")
with open('output/q4_feature_list.txt', 'w') as f:
    for feature in new_features:
        f.write(f"{feature}\n")

print("✓ Saved: output/q4_feature_list.txt")
print(f"  Total features listed: {len(new_features)}")


SAVING ARTIFACTS

Saving all features...
✓ Saved: output/q4_features.csv
  Columns saved: 42
  Rows saved: 195892

Saving rolling features...
✓ Saved: output/q4_rolling_features.csv
  Columns saved: 8
  Rolling features: 7

Sample of rolling features:
  Measurement Timestamp  wind_speed_rolling_7h  wind_speed_rolling_24h  \
0   2015-04-25 09:00:00               5.100000                5.100000   
1   2015-04-30 05:00:00               6.150000                6.150000   
2   2015-05-22 15:00:00               4.733333                4.733333   
3   2015-05-22 16:00:00               4.550000                4.550000   
4   2015-05-22 17:00:00               3.880000                3.880000   

   wind_speed_rolling_std_7h  humidity_rolling_7h  humidity_rolling_24h  \
0                   1.484924            86.000000             86.000000   
1                   1.484924            81.000000             81.000000   
2                   2.668957            72.333333             72.333333   
3 

In [12]:

# ========================================
# VERIFICATION
# ========================================
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

print("\nVerifying feature creation:")
print(f"  Derived features: {len(new_features) - len(rolling_features) - len(categorical_features)}")
print(f"  Rolling features: {len(rolling_features)}")
print(f"  Categorical features: {len(categorical_features)}")
print(f"  Total new features: {len(new_features)}")

print("\nNew features created:")
for i, feature in enumerate(new_features, 1):
    print(f"  {i}. {feature}")

# Verify files exist
print("\nVerifying output files:")
output_files = [
    'output/q4_features.csv',
    'output/q4_rolling_features.csv',
    'output/q4_feature_list.txt'
]
for file in output_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # KB
        print(f"  ✓ {file} ({size:.1f} KB)")
    else:
        print(f"  ✗ {file} NOT FOUND!")

# Check for any remaining issues
print("\nFinal data quality check:")
total_missing = df.isnull().sum().sum()
print(f"  Total missing values: {total_missing}")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {df.shape[1]}")

# ========================================
# SUMMARY
# ========================================
print("\n" + "="*60)
print("Q4 COMPLETE - All artifacts created successfully!")
print("="*60)
print("\nFiles created:")
print("  1. output/q4_features.csv")
print("  2. output/q4_rolling_features.csv")
print("  3. output/q4_feature_list.txt")
print("\nFeature Engineering Summary:")
print(f"  - Original columns: {df.shape[1] - len(new_features)}")
print(f"  - New features: {len(new_features)}")
print(f"  - Total columns: {df.shape[1]}")
print(f"  - Derived features: {len(new_features) - len(rolling_features) - len(categorical_features)}")
print(f"  - Rolling features: {len(rolling_features)}")
print(f"  - Categorical features: {len(categorical_features)}")
print("\nNext: Proceed to Q5 for pattern analysis")
print("="*60)


VERIFICATION

Verifying feature creation:
  Derived features: 6
  Rolling features: 7
  Categorical features: 3
  Total new features: 16

New features created:
  1. wind_speed_squared
  2. wind_category
  3. comfort_index
  4. humidity_squared
  5. temp_wind_interaction
  6. pressure_deviation
  7. wind_speed_rolling_7h
  8. wind_speed_rolling_24h
  9. wind_speed_rolling_std_7h
  10. humidity_rolling_7h
  11. humidity_rolling_24h
  12. pressure_rolling_7h
  13. pressure_rolling_24h
  14. temp_category
  15. time_of_day
  16. season

Verifying output files:
  ✓ output/q4_features.csv (72884.5 KB)
  ✓ output/q4_rolling_features.csv (26430.3 KB)
  ✓ output/q4_feature_list.txt (0.3 KB)

Final data quality check:
  Total missing values: 379272
  Total rows: 195,892
  Total columns: 41

Q4 COMPLETE - All artifacts created successfully!

Files created:
  1. output/q4_features.csv
  2. output/q4_rolling_features.csv
  3. output/q4_feature_list.txt

Feature Engineering Summary:
  - Original co